# Phase 2: Structural Inferential Analysis

## Overview
Statistical hypothesis testing on circuit structural properties across frequency bands
and model scales. Tests whether structural differences between frequency-band circuits
are statistically significant.

## Hypothesis Framework
11 domains, ~80 tests total, with BH-FDR correction:

- **D1: Circuit Size**: Do edge counts/fractions differ by band?
- **D2: Edge Categories**: Do skip/local/forward proportions differ?
- **D3: Head Participation**: Do head participation rates differ by band?
- **D4: Component Composition**: Do attn/mlp/resid ratios differ?
- **D5: Universal Edges**: Do universal edge fractions differ by model?
- **D6: Jaccard Similarity**: Within-band > between-band similarity?
- **D7: Model Scaling**: Do structural metrics scale with model size?
- **D8: Control vs Frequency**: Does control circuit structure differ?
- **D9: Variance Decomposition**: Band vs Model vs Draw contributions?
- **D10: Draw Stability**: Are structural metrics reliable across draws?
- **D11: Structure-Function**: Do structural properties predict functional performance?

## Data Sources
- Extracted structural data from `outputs/extraction/all_circuits_structure.json`
- Phase 1 functional data for structure-function linkage

## Notebook Structure
1. Setup & Data Loading
2. D1: Circuit Size by Band
3. D2: Edge Categories by Band
4. D3: Head Participation
5. D4: Component Composition
6. D5: Universal Edges
7. D6: Jaccard Similarity
8. D7: Model Scaling
9. D8: Control vs Frequency
10. D9: Variance Decomposition
11. D10: Draw Stability
12. D11: Structure-Function Linkage
13. Multiple Comparison Correction & Summary
14. Visualizations

## 1. Setup & Data Loading

In [1]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from scipy import stats as sp_stats

# Add Phase 2 project path (Phase 1 stats re-exported via utils/stats.py)
sys.path.insert(0, "LSC_circuit_analysis/02_Phase_Structural")

from utils.constants import (
    MODELS,
    BANDS,
    DRAWS,
    FREQUENCY_BANDS,
    FREQUENCY_RANK,
    MODEL_LAYERS,
    MODEL_HEADS,
    MODEL_INFO,
    MODEL_CAPACITY,
    BAND_COLORS,
    MODEL_COLORS,
    BAND_NAMES,
    ANALYSIS_DIR,
    VIZ_DIR,
    get_output_dirs,
    ALPHA,
)
from utils.data_loading import load_extracted_data, load_functional_data
from utils.edge_analysis import (
    compute_within_between_jaccard,
    compute_universal_edges_per_draw,
)
from utils.plotting import setup_plotting, save_figure

# Phase 1 stats (re-exported via Phase 2's utils/stats.py)
from utils.stats import (
    TestAccumulator,
    safe_kruskal,
    safe_mannwhitneyu,
    safe_wilcoxon,
    safe_spearmanr,
    jonckheere_terpstra,
    cohens_d,
    rank_biserial,
    eta_squared,
    bootstrap_ci,
    bootstrap_ci_diff,
)

setup_plotting()
ANALYSIS_DIR, VIZ_DIR = get_output_dirs()

# Initialize test accumulator
acc = TestAccumulator()

In [2]:
# Load structural data
circuits, df = load_extracted_data()
print(f"Structural data: {df.shape}")

# Load functional data for S-F linkage
df_func = load_functional_data()
print(f"Functional data: {df_func.shape}")

# Merge structural + functional on model, band, draw
df_merged = df.merge(
    df_func[
        [
            "model",
            "band",
            "draw",
            "retention_ratio",
            "completeness",
            "circuit_accuracy",
            "base_accuracy",
            "circuit_kl_div",
            "size_fraction",
            "top1_top5_gap",
        ]
    ],
    on=["model", "band", "draw"],
    how="left",
)
print(f"Merged data: {df_merged.shape}")

Loaded 75 circuits (0 failed)
Structural data: (75, 30)


Loaded 75 / 75 metrics files
Functional data: (75, 27)
Merged data: (75, 37)


## 2. D1: Circuit Size by Band

In [3]:
# D1: Do edge counts/fractions differ by frequency band?
# Kruskal-Wallis per model + pairwise Mann-Whitney

for model in MODELS:
    subset = df[df["model"] == model]

    for metric, metric_name in [
        ("total_edges", "total_edges"),
        ("edge_fraction", "edge_fraction"),
    ]:
        # Omnibus: Kruskal-Wallis across all 5 bands
        groups = [subset[subset["band"] == b][metric].values for b in BANDS]
        H, p = safe_kruskal(*groups)
        es = eta_squared(groups)

        acc.add_test(
            domain="D1_Circuit_Size",
            hypothesis=f"{metric_name} differs by band",
            model=model,
            test_name="Kruskal-Wallis",
            comparison="omnibus_5bands",
            statistic=H,
            p_value=p,
            effect_size=es,
            effect_type="eta_squared",
            n1=sum(len(g) for g in groups),
            n2=len(groups),
        )

        # Pairwise: freq bands only (low vs medium, low vs high, etc.)
        for i, b1 in enumerate(FREQUENCY_BANDS):
            for b2 in FREQUENCY_BANDS[i + 1 :]:
                g1 = subset[subset["band"] == b1][metric].values
                g2 = subset[subset["band"] == b2][metric].values
                U, p_pw = safe_mannwhitneyu(g1, g2)
                r = rank_biserial(g1, g2)
                ci_l, ci_u, _ = bootstrap_ci_diff(g1, g2)

                acc.add_test(
                    domain="D1_Circuit_Size",
                    hypothesis=f"{metric_name}: {b1} vs {b2}",
                    model=model,
                    test_name="Mann-Whitney U",
                    comparison=f"{b1}_vs_{b2}",
                    statistic=U,
                    p_value=p_pw,
                    effect_size=r,
                    effect_type="rank_biserial",
                    n1=len(g1),
                    n2=len(g2),
                    ci_lower=ci_l,
                    ci_upper=ci_u,
                )

print(f"D1 tests: {len([t for t in acc.tests if t['domain'] == 'D1_Circuit_Size'])}")

D1 tests: 70


## 3. D2: Edge Categories by Band

In [4]:
# D2: Do edge category proportions differ by band?
category_metrics = [
    "skip_fraction",
    "local_fraction",
    "forward_fraction",
    "input_fraction",
]

for model in MODELS:
    subset = df[df["model"] == model]

    for metric in category_metrics:
        groups = [subset[subset["band"] == b][metric].values for b in BANDS]
        H, p = safe_kruskal(*groups)
        es = eta_squared(groups)

        acc.add_test(
            domain="D2_Edge_Categories",
            hypothesis=f"{metric} differs by band",
            model=model,
            test_name="Kruskal-Wallis",
            comparison="omnibus_5bands",
            statistic=H,
            p_value=p,
            effect_size=es,
            effect_type="eta_squared",
            n1=sum(len(g) for g in groups),
            n2=len(groups),
        )

print(f"D2 tests: {len([t for t in acc.tests if t['domain'] == 'D2_Edge_Categories'])}")

D2 tests: 20


## 4. D3: Head Participation

In [5]:
# D3: Do head participation rates differ by band?

for model in MODELS:
    subset = df[df["model"] == model]

    for metric in ["head_participation_rate", "mean_edges_per_head"]:
        groups = [subset[subset["band"] == b][metric].values for b in BANDS]
        H, p = safe_kruskal(*groups)
        es = eta_squared(groups)

        acc.add_test(
            domain="D3_Head_Participation",
            hypothesis=f"{metric} differs by band",
            model=model,
            test_name="Kruskal-Wallis",
            comparison="omnibus_5bands",
            statistic=H,
            p_value=p,
            effect_size=es,
            effect_type="eta_squared",
            n1=sum(len(g) for g in groups),
            n2=len(groups),
        )

print(
    f"D3 tests: {len([t for t in acc.tests if t['domain'] == 'D3_Head_Participation'])}"
)

D3 tests: 10


## 5. D4: Component Composition

In [6]:
# D4: Do component ratios differ by band?

for model in MODELS:
    subset = df[df["model"] == model]

    for metric in ["attn_fraction", "mlp_fraction", "resid_fraction"]:
        groups = [subset[subset["band"] == b][metric].values for b in BANDS]
        H, p = safe_kruskal(*groups)
        es = eta_squared(groups)

        acc.add_test(
            domain="D4_Component_Composition",
            hypothesis=f"{metric} differs by band",
            model=model,
            test_name="Kruskal-Wallis",
            comparison="omnibus_5bands",
            statistic=H,
            p_value=p,
            effect_size=es,
            effect_type="eta_squared",
            n1=sum(len(g) for g in groups),
            n2=len(groups),
        )

print(
    f"D4 tests: {len([t for t in acc.tests if t['domain'] == 'D4_Component_Composition'])}"
)

D4 tests: 15


## 6. D5: Universal Edges

In [7]:
# D5: Do universal edge fractions differ by model?
# Also: is the universal fraction significantly above chance?

# Compute universal fractions per draw
univ_data = []
for model in MODELS:
    per_draw = compute_universal_edges_per_draw(circuits, model)
    model_circuits = [c for c in circuits.values() if c["model"] == model]

    for draw, edges in per_draw.items():
        draw_circuits = [c for c in model_circuits if c["draw"] == draw]
        mean_size = np.mean([c["total_edges"] for c in draw_circuits])
        frac = len(edges) / mean_size if mean_size > 0 else 0
        univ_data.append({"model": model, "draw": draw, "universal_fraction": frac})

df_univ = pd.DataFrame(univ_data)

# KW test: universal fraction across models
groups = [df_univ[df_univ["model"] == m]["universal_fraction"].values for m in MODELS]
H, p = safe_kruskal(*groups)
es = eta_squared(groups)

acc.add_test(
    domain="D5_Universal_Edges",
    hypothesis="Universal edge fraction differs by model",
    model="all",
    test_name="Kruskal-Wallis",
    comparison="omnibus_4models",
    statistic=H,
    p_value=p,
    effect_size=es,
    effect_type="eta_squared",
    n1=sum(len(g) for g in groups),
    n2=len(groups),
)

# Pairwise model comparisons
for i, m1 in enumerate(MODELS):
    for m2 in MODELS[i + 1 :]:
        g1 = df_univ[df_univ["model"] == m1]["universal_fraction"].values
        g2 = df_univ[df_univ["model"] == m2]["universal_fraction"].values
        U, p_pw = safe_mannwhitneyu(g1, g2)
        r = rank_biserial(g1, g2)

        acc.add_test(
            domain="D5_Universal_Edges",
            hypothesis=f"Universal fraction: {m1} vs {m2}",
            model="pairwise",
            test_name="Mann-Whitney U",
            comparison=f"{m1}_vs_{m2}",
            statistic=U,
            p_value=p_pw,
            effect_size=r,
            effect_type="rank_biserial",
            n1=len(g1),
            n2=len(g2),
        )

print(f"D5 tests: {len([t for t in acc.tests if t['domain'] == 'D5_Universal_Edges'])}")

D5 tests: 11


## 7. D6: Jaccard Similarity

In [8]:
# D6: Is within-band Jaccard > between-band Jaccard?

for model in MODELS:
    wb = compute_within_between_jaccard(circuits, model)
    within = np.array(wb["within"])
    between = np.array(wb["between"])

    if len(within) > 0 and len(between) > 0:
        U, p = safe_mannwhitneyu(within, between, alternative="greater")
        r = rank_biserial(within, between)
        d = cohens_d(within, between)
        ci_l, ci_u, diff = bootstrap_ci_diff(within, between)

        acc.add_test(
            domain="D6_Jaccard_Similarity",
            hypothesis="Within-band > between-band Jaccard",
            model=model,
            test_name="Mann-Whitney U (one-sided)",
            comparison="within_vs_between",
            statistic=U,
            p_value=p,
            effect_size=d,
            effect_type="cohens_d",
            n1=len(within),
            n2=len(between),
            ci_lower=ci_l,
            ci_upper=ci_u,
        )

# Supplementary: same-draw-only between-band Jaccard
# The default between-band pool mixes same-draw and cross-draw pairs,
# which adds draw variability to the between-band estimates. This makes
# the within > between comparison more conservative (a strength), but
# same-draw-only between-band Jaccard isolates the pure band effect.
from utils.edge_analysis import get_edge_set, compute_jaccard

print("\nD6 same-draw-only between-band Jaccard (supplementary):")
for model in MODELS:
    band_draw_edges = defaultdict(dict)
    for c in circuits.values():
        if c["model"] == model and c["band"] in BANDS:
            band_draw_edges[c["band"]][c["draw"]] = get_edge_set(c)

    # Same-draw between-band: compare different bands within the same draw
    same_draw_between = []
    band_list = [b for b in BANDS if b in band_draw_edges]
    for draw in DRAWS:
        for i, b1 in enumerate(band_list):
            for j, b2 in enumerate(band_list):
                if i >= j:
                    continue
                if draw in band_draw_edges[b1] and draw in band_draw_edges[b2]:
                    jac = compute_jaccard(
                        band_draw_edges[b1][draw], band_draw_edges[b2][draw]
                    )
                    same_draw_between.append(jac)

    wb = compute_within_between_jaccard(circuits, model)
    within = np.array(wb["within"])
    between_all = np.array(wb["between"])
    same_draw_between = np.array(same_draw_between)

    if len(within) > 0 and len(same_draw_between) > 0:
        U, p = safe_mannwhitneyu(within, same_draw_between, alternative="greater")
        d = cohens_d(within, same_draw_between)
        ci_l, ci_u, diff = bootstrap_ci_diff(within, same_draw_between)

        acc.add_test(
            domain="D6_Jaccard_Similarity",
            hypothesis="Within-band > between-band Jaccard (same-draw only)",
            model=model,
            test_name="Mann-Whitney U (one-sided)",
            comparison="within_vs_between_same_draw",
            statistic=U,
            p_value=p,
            effect_size=d,
            effect_type="cohens_d",
            n1=len(within),
            n2=len(same_draw_between),
            ci_lower=ci_l,
            ci_upper=ci_u,
        )

        print(
            f"  {model}: within={within.mean():.4f} (n={len(within)}), "
            f"between_same_draw={same_draw_between.mean():.4f} (n={len(same_draw_between)}), "
            f"between_all={between_all.mean():.4f} (n={len(between_all)}), "
            f"d={d:.3f}, p={p:.4f}"
        )

print(
    f"\nD6 tests: {len([t for t in acc.tests if t['domain'] == 'D6_Jaccard_Similarity'])}"
)


D6 same-draw-only between-band Jaccard (supplementary):


  pythia-70m: within=0.7950 (n=30), between_same_draw=0.7584 (n=30), between_all=0.7626 (n=90), d=1.511, p=0.0000


  pythia-160m: within=0.5891 (n=30), between_same_draw=0.5628 (n=30), between_all=0.5568 (n=90), d=1.029, p=0.0000


  pythia-410m: within=0.4463 (n=30), between_same_draw=0.4321 (n=30), between_all=0.4304 (n=90), d=0.914, p=0.0001


  pythia-1b: within=0.4779 (n=30), between_same_draw=0.4657 (n=30), between_all=0.4651 (n=90), d=0.512, p=0.0259


  pythia-1.4b: within=0.3848 (n=30), between_same_draw=0.3615 (n=30), between_all=0.3661 (n=90), d=0.964, p=0.0003

D6 tests: 10


## 8. D7: Model Scaling

In [9]:
# D7: Do structural metrics scale with model size?
# Jonckheere-Terpstra trend test + Spearman correlation

scaling_metrics = [
    "edge_fraction",
    "skip_fraction",
    "head_participation_rate",
    "attn_fraction",
    "mlp_fraction",
]

for metric in scaling_metrics:
    # JT trend test (ordered by model capacity)
    groups = [df[df["model"] == m][metric].values for m in MODELS]
    Z, p_jt = jonckheere_terpstra(groups, alternative="two-sided")

    acc.add_test(
        domain="D7_Model_Scaling",
        hypothesis=f"{metric} trends with model size",
        model="all",
        test_name="Jonckheere-Terpstra",
        comparison="trend_4models",
        statistic=Z,
        p_value=p_jt,
        effect_size=Z,
        effect_type="Z_statistic",
        n1=sum(len(g) for g in groups),
        n2=len(groups),
    )

    # Spearman correlation with model capacity
    capacities = df["model"].map(MODEL_CAPACITY).values
    rho, p_sp = safe_spearmanr(capacities, df[metric].values)

    acc.add_test(
        domain="D7_Model_Scaling",
        hypothesis=f"{metric} correlates with model capacity",
        model="all",
        test_name="Spearman",
        comparison="capacity_correlation",
        statistic=rho,
        p_value=p_sp,
        effect_size=rho,
        effect_type="spearman_rho",
        n1=len(df),
        n2=0,
    )

print(f"D7 tests: {len([t for t in acc.tests if t['domain'] == 'D7_Model_Scaling'])}")

D7 tests: 10


## 9. D8: Control vs Frequency Bands

In [10]:
# D8: Does control circuit structure differ from frequency bands?

control_metrics = [
    "edge_fraction",
    "skip_fraction",
    "head_participation_rate",
    "attn_fraction",
    "mlp_fraction",
]

for model in MODELS:
    subset = df[df["model"] == model]
    control = subset[subset["band"] == "control"]
    freq = subset[subset["band"].isin(FREQUENCY_BANDS)]

    for metric in control_metrics:
        g1 = control[metric].values
        g2 = freq[metric].values
        U, p = safe_mannwhitneyu(g1, g2)
        r = rank_biserial(g1, g2)

        acc.add_test(
            domain="D8_Control_vs_Freq",
            hypothesis=f"{metric}: control vs frequency bands",
            model=model,
            test_name="Mann-Whitney U",
            comparison="control_vs_freq",
            statistic=U,
            p_value=p,
            effect_size=r,
            effect_type="rank_biserial",
            n1=len(g1),
            n2=len(g2),
        )

print(f"D8 tests: {len([t for t in acc.tests if t['domain'] == 'D8_Control_vs_Freq'])}")

D8 tests: 25


## 10. D9: Variance Decomposition

In [11]:
# D9: What proportion of variance is explained by Band vs Model vs Draw?
# Unified ANOVA: metric ~ C(model) + C(band) + C(draw) + C(model):C(band)

from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

variance_metrics = ["edge_fraction", "skip_fraction", "head_participation_rate"]

var_rows = []
for metric in variance_metrics:
    df_anova = df[["model", "band", "draw", metric]].dropna()
    df_anova.columns = ["model", "band", "draw", "y"]

    formula = "y ~ C(model) + C(band) + C(draw) + C(model):C(band)"
    model_fit = ols(formula, data=df_anova).fit()
    anova_table = anova_lm(model_fit, typ=2)

    ss_total = anova_table["sum_sq"].sum()

    for factor in ["C(model)", "C(band)", "C(draw)", "C(model):C(band)"]:
        if factor in anova_table.index:
            ss = anova_table.loc[factor, "sum_sq"]
            f_val = anova_table.loc[factor, "F"]
            p_val = anova_table.loc[factor, "PR(>F)"]
            eta2 = ss / ss_total

            factor_clean = factor.replace("C(", "").replace(")", "").replace(":", "_x_")

            var_rows.append(
                {
                    "metric": metric,
                    "factor": factor_clean,
                    "SS": ss,
                    "eta_squared": eta2,
                    "F": f_val,
                    "p_value": p_val,
                }
            )

            acc.add_test(
                domain="D9_Variance_Decomposition",
                hypothesis=f"{metric}: {factor_clean} effect",
                model="all",
                test_name="ANOVA",
                comparison=factor_clean,
                statistic=f_val,
                p_value=p_val,
                effect_size=eta2,
                effect_type="eta_squared",
                n1=len(df_anova),
                n2=0,
            )

    # Residual
    resid_ss = anova_table.loc["Residual", "sum_sq"]
    var_rows.append(
        {
            "metric": metric,
            "factor": "Residual",
            "SS": resid_ss,
            "eta_squared": resid_ss / ss_total,
            "F": np.nan,
            "p_value": np.nan,
        }
    )

df_var = pd.DataFrame(var_rows)
print("Variance Decomposition:")
for metric in variance_metrics:
    sub = df_var[df_var["metric"] == metric]
    print(f"\n  {metric}:")
    for _, row in sub.iterrows():
        sig = (
            "*"
            if row["p_value"] < ALPHA
            else ""
            if not np.isnan(row["p_value"])
            else ""
        )
        print(
            f"    {row['factor']:>20}: eta2={row['eta_squared']:.4f} ({row['eta_squared']:.1%}){sig}"
        )
    total = sub["eta_squared"].sum()
    print(f"    {'Sum':>20}: {total:.4f}")

print(
    f"\nD9 tests: {len([t for t in acc.tests if t['domain'] == 'D9_Variance_Decomposition'])}"
)

Variance Decomposition:

  edge_fraction:
                   model: eta2=0.9956 (99.6%)*
                    band: eta2=0.0002 (0.0%)
                    draw: eta2=0.0000 (0.0%)
            model_x_band: eta2=0.0031 (0.3%)*
                Residual: eta2=0.0012 (0.1%)
                     Sum: 1.0000

  skip_fraction:
                   model: eta2=0.9922 (99.2%)*
                    band: eta2=0.0023 (0.2%)*
                    draw: eta2=0.0001 (0.0%)
            model_x_band: eta2=0.0021 (0.2%)*
                Residual: eta2=0.0034 (0.3%)
                     Sum: 1.0000

  head_participation_rate:
                   model: eta2=0.9622 (96.2%)*
                    band: eta2=0.0046 (0.5%)
                    draw: eta2=0.0001 (0.0%)
            model_x_band: eta2=0.0113 (1.1%)
                Residual: eta2=0.0217 (2.2%)
                     Sum: 1.0000

D9 tests: 12


## 11. D10: Draw Stability

In [12]:
# D10: Are structural metrics reliable across draws?
# ICC (intraclass correlation) and coefficient of variation

stability_metrics = ["edge_fraction", "skip_fraction", "head_participation_rate"]

stab_rows = []
for model in MODELS:
    for band in BANDS:
        subset = df[(df["model"] == model) & (df["band"] == band)]
        for metric in stability_metrics:
            vals = subset[metric].values
            if len(vals) >= 2:
                cv = np.std(vals) / np.mean(vals) if np.mean(vals) != 0 else np.nan
                stab_rows.append(
                    {
                        "model": model,
                        "band": band,
                        "metric": metric,
                        "mean": np.mean(vals),
                        "std": np.std(vals),
                        "cv": cv,
                    }
                )

df_stab = pd.DataFrame(stab_rows)

# Summary: mean CV per metric across models/bands
for metric in stability_metrics:
    sub = df_stab[df_stab["metric"] == metric]
    print(f"{metric}: mean CV = {sub['cv'].mean():.3f} +/- {sub['cv'].std():.3f}")

    # Add as test result
    acc.add_test(
        domain="D10_Draw_Stability",
        hypothesis=f"{metric} is stable across draws",
        model="all",
        test_name="Coefficient of Variation",
        comparison="mean_CV",
        statistic=sub["cv"].mean(),
        p_value=np.nan,  # CV doesn't have a p-value
        effect_size=sub["cv"].mean(),
        effect_type="CV",
        n1=len(sub),
        n2=0,
    )

print(
    f"\nD10 tests: {len([t for t in acc.tests if t['domain'] == 'D10_Draw_Stability'])}"
)

edge_fraction: mean CV = 0.026 +/- 0.014
skip_fraction: mean CV = 0.006 +/- 0.003
head_participation_rate: mean CV = 0.029 +/- 0.015

D10 tests: 3


## 12. D11: Structure-Function Linkage

In [13]:
# D11: Do structural properties predict functional performance?
# Spearman correlations between structural and functional metrics

struct_metrics = [
    "edge_fraction",
    "skip_fraction",
    "head_participation_rate",
    "attn_fraction",
    "total_edges",
]
func_metrics = ["retention_ratio", "circuit_accuracy", "circuit_kl_div"]

sf_rows = []
for s_metric in struct_metrics:
    for f_metric in func_metrics:
        # Overall correlation
        valid = df_merged[[s_metric, f_metric]].dropna()
        if len(valid) < 5:
            continue

        rho, p = safe_spearmanr(valid[s_metric].values, valid[f_metric].values)

        acc.add_test(
            domain="D11_Structure_Function",
            hypothesis=f"{s_metric} correlates with {f_metric}",
            model="all",
            test_name="Spearman",
            comparison=f"{s_metric}_vs_{f_metric}",
            statistic=rho,
            p_value=p,
            effect_size=rho,
            effect_type="spearman_rho",
            n1=len(valid),
            n2=0,
        )

        sf_rows.append(
            {
                "structural": s_metric,
                "functional": f_metric,
                "rho": rho,
                "p_value": p,
                "n": len(valid),
                "significant": p < ALPHA if not np.isnan(p) else False,
                "model": "all",
            }
        )

df_sf = pd.DataFrame(sf_rows)
print("Structure-Function Correlations (overall, n=60):")
print(
    df_sf[["structural", "functional", "rho", "p_value", "significant"]].to_string(
        index=False
    )
)

# Within-model correlations (controlling for between-model variation)
# NOTE: The overall D11 correlations are largely driven by between-model
# variation (model explains ~99% of edge_fraction variance). Within-model
# correlations test whether structural variation WITHIN a model predicts
# functional performance, which is the more informative question.
print("\n\nWithin-Model Structure-Function Correlations (n=15 per model):")
sf_within_rows = []
for model in MODELS:
    model_data = df_merged[df_merged["model"] == model]
    for s_metric in struct_metrics:
        for f_metric in func_metrics:
            valid = model_data[[s_metric, f_metric]].dropna()
            if len(valid) < 5:
                continue
            rho, p = safe_spearmanr(valid[s_metric].values, valid[f_metric].values)

            acc.add_test(
                domain="D11_Structure_Function_Within",
                hypothesis=f"{s_metric} correlates with {f_metric} (within-model)",
                model=model,
                test_name="Spearman",
                comparison=f"{s_metric}_vs_{f_metric}_within",
                statistic=rho,
                p_value=p,
                effect_size=rho,
                effect_type="spearman_rho",
                n1=len(valid),
                n2=0,
            )

            sf_within_rows.append(
                {
                    "model": model,
                    "structural": s_metric,
                    "functional": f_metric,
                    "rho": rho,
                    "p_value": p,
                    "n": len(valid),
                    "significant": p < ALPHA if not np.isnan(p) else False,
                }
            )

df_sf_within = pd.DataFrame(sf_within_rows)

# Show summary: how many within-model correlations are significant per model?
for model in MODELS:
    sub = df_sf_within[df_sf_within["model"] == model]
    n_sig = sub["significant"].sum()
    print(f"  {model}: {n_sig}/{len(sub)} significant (uncorrected)")
    # Show top correlations
    top = sub.nlargest(3, "rho", keep="first")
    for _, row in top.iterrows():
        sig_mark = "*" if row["significant"] else ""
        print(
            f"    {row['structural']:>25} vs {row['functional']:<20} rho={row['rho']:+.3f} p={row['p_value']:.4f}{sig_mark}"
        )

# Save both tables
sf_all = pd.concat(
    [
        df_sf.assign(scope="overall"),
        df_sf_within.assign(scope="within_model"),
    ],
    ignore_index=True,
)
sf_all.to_csv(ANALYSIS_DIR / "structure_function_correlations.csv", index=False)

print(
    f"\nD11 tests (overall): {len([t for t in acc.tests if t['domain'] == 'D11_Structure_Function'])}"
)
print(
    f"D11 tests (within-model): {len([t for t in acc.tests if t['domain'] == 'D11_Structure_Function_Within'])}"
)

Structure-Function Correlations (overall, n=60):
             structural       functional       rho      p_value  significant
          edge_fraction  retention_ratio -0.240785 3.743748e-02         True
          edge_fraction circuit_accuracy -0.460402 3.236427e-05         True
          edge_fraction   circuit_kl_div -0.707814 1.241393e-12         True
          skip_fraction  retention_ratio  0.530575 9.765553e-07         True
          skip_fraction circuit_accuracy  0.607502 7.506534e-09         True
          skip_fraction   circuit_kl_div  0.375135 9.125261e-04         True
head_participation_rate  retention_ratio -0.235480 4.197537e-02         True
head_participation_rate circuit_accuracy -0.436261 9.130540e-05         True
head_participation_rate   circuit_kl_div -0.702745 2.100931e-12         True
          attn_fraction  retention_ratio -0.394805 4.562235e-04         True
          attn_fraction circuit_accuracy -0.206278 7.580527e-02        False
          attn_fraction   c

  pythia-70m: 4/15 significant (uncorrected)
                edge_fraction vs circuit_accuracy     rho=+0.771 p=0.0008*
                  total_edges vs circuit_accuracy     rho=+0.771 p=0.0008*
                attn_fraction vs circuit_accuracy     rho=+0.695 p=0.0040*
  pythia-160m: 4/15 significant (uncorrected)
      head_participation_rate vs circuit_kl_div       rho=+0.355 p=0.1940
                edge_fraction vs circuit_kl_div       rho=+0.225 p=0.4201
                  total_edges vs circuit_kl_div       rho=+0.225 p=0.4201
  pythia-410m: 0/15 significant (uncorrected)
      head_participation_rate vs circuit_kl_div       rho=+0.333 p=0.2247
                edge_fraction vs circuit_kl_div       rho=+0.207 p=0.4588
                  total_edges vs circuit_kl_div       rho=+0.207 p=0.4588
  pythia-1b: 0/15 significant (uncorrected)
                edge_fraction vs circuit_kl_div       rho=+0.421 p=0.1177
                  total_edges vs circuit_kl_div       rho=+0.421 p=0.1177
  

## 13. Multiple Comparison Correction & Summary

In [14]:
# Collect all tests and apply BH-FDR correction
df_tests = acc.to_dataframe()
print(f"Total tests: {len(df_tests)}")
print(f"Tests by domain:")
print(df_tests["domain"].value_counts().to_string())

# Apply FDR correction
df_tests = acc.apply_fdr_correction(df_tests)

# Summary
print("\n" + "=" * 70)
print("RESULTS SUMMARY")
print("=" * 70)
print(acc.summary(df_tests))

# Save
df_tests.to_csv(ANALYSIS_DIR / "all_hypothesis_tests.csv", index=False)
print(f"\nSaved: {ANALYSIS_DIR / 'all_hypothesis_tests.csv'}")

Total tests: 276
Tests by domain:
domain
D11_Structure_Function_Within    75
D1_Circuit_Size                  70
D8_Control_vs_Freq               25
D2_Edge_Categories               20
D4_Component_Composition         15
D11_Structure_Function           15
D9_Variance_Decomposition        12
D5_Universal_Edges               11
D3_Head_Participation            10
D6_Jaccard_Similarity            10
D7_Model_Scaling                 10
D10_Draw_Stability                3

RESULTS SUMMARY
D1_Circuit_Size: 0/70 significant
D2_Edge_Categories: 0/20 significant
D3_Head_Participation: 0/10 significant
D4_Component_Composition: 0/15 significant
D5_Universal_Edges: 0/11 significant
D6_Jaccard_Similarity: 9/10 significant
D7_Model_Scaling: 4/10 significant
D8_Control_vs_Freq: 0/25 significant
D9_Variance_Decomposition: 5/12 significant
D10_Draw_Stability: 0/3 significant
D11_Structure_Function: 12/15 significant
D11_Structure_Function_Within: 12/75 significant

Saved: LSC_circuit_analysis/02_Phas

In [15]:
# Detailed results by domain
print("\n" + "=" * 70)
print("SIGNIFICANT RESULTS (BH-FDR corrected)")
print("=" * 70)

sig_col = (
    "significant_bh"
    if "significant_bh" in df_tests.columns
    else "significant_uncorrected"
)

for domain in df_tests["domain"].unique():
    d = df_tests[df_tests["domain"] == domain]
    n_sig = d[sig_col].sum()
    print(f"\n--- {domain}: {n_sig}/{len(d)} significant ---")

    sig_tests = d[d[sig_col]]
    if len(sig_tests) > 0:
        for _, row in sig_tests.iterrows():
            p_col = "p_value_bh" if "p_value_bh" in row.index else "p_value"
            print(
                f"  {row['hypothesis']}: "
                f"p={row[p_col]:.4f}, "
                f"effect={row['effect_size']:.3f} ({row['effect_type']}), "
                f"model={row['model']}"
            )
    else:
        print("  (none)")


SIGNIFICANT RESULTS (BH-FDR corrected)

--- D1_Circuit_Size: 0/70 significant ---
  (none)

--- D2_Edge_Categories: 0/20 significant ---
  (none)

--- D3_Head_Participation: 0/10 significant ---
  (none)

--- D4_Component_Composition: 0/15 significant ---


  (none)

--- D5_Universal_Edges: 0/11 significant ---
  (none)

--- D6_Jaccard_Similarity: 9/10 significant ---
  Within-band > between-band Jaccard: p=0.0000, effect=1.123 (cohens_d), model=pythia-70m
  Within-band > between-band Jaccard: p=0.0000, effect=1.373 (cohens_d), model=pythia-160m
  Within-band > between-band Jaccard: p=0.0000, effect=0.980 (cohens_d), model=pythia-410m
  Within-band > between-band Jaccard: p=0.0150, effect=0.528 (cohens_d), model=pythia-1b
  Within-band > between-band Jaccard: p=0.0067, effect=0.738 (cohens_d), model=pythia-1.4b
  Within-band > between-band Jaccard (same-draw only): p=0.0000, effect=1.511 (cohens_d), model=pythia-70m
  Within-band > between-band Jaccard (same-draw only): p=0.0002, effect=1.029 (cohens_d), model=pythia-160m
  Within-band > between-band Jaccard (same-draw only): p=0.0016, effect=0.914 (cohens_d), model=pythia-410m
  Within-band > between-band Jaccard (same-draw only): p=0.0028, effect=0.964 (cohens_d), model=pythia-1.4b

---

In [16]:
# Save auxiliary tables
df_var.to_csv(ANALYSIS_DIR / "variance_decomposition.csv", index=False)
# structure_function_correlations.csv already saved in D11 cell (now includes within-model)
df_stab.to_csv(ANALYSIS_DIR / "draw_stability.csv", index=False)

print("Saved auxiliary tables:")
print(f"  {ANALYSIS_DIR / 'variance_decomposition.csv'}")
print(f"  {ANALYSIS_DIR / 'structure_function_correlations.csv'} (saved in D11 cell)")
print(f"  {ANALYSIS_DIR / 'draw_stability.csv'}")

Saved auxiliary tables:
  LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/variance_decomposition.csv
  LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/structure_function_correlations.csv (saved in D11 cell)
  LSC_circuit_analysis/02_Phase_Structural/outputs/analysis/draw_stability.csv


## 14. Visualizations

In [17]:
# VIZ 14: Significance heatmap by domain x model
fig, ax = plt.subplots(figsize=(14, 8))

# Build matrix: domains x models, value = fraction of significant tests
domains = [d for d in df_tests["domain"].unique() if d != "D10_Draw_Stability"]
models_plus = MODELS + ["all", "pairwise"]

sig_matrix = []
domain_labels = []
for domain in domains:
    row_vals = []
    for model in models_plus:
        subset = df_tests[(df_tests["domain"] == domain) & (df_tests["model"] == model)]
        if len(subset) > 0:
            frac = subset[sig_col].mean()
        else:
            frac = np.nan
        row_vals.append(frac)
    if not all(np.isnan(v) for v in row_vals):
        sig_matrix.append(row_vals)
        domain_labels.append(domain.replace("_", " "))

sig_arr = np.array(sig_matrix)
sns.heatmap(
    sig_arr,
    ax=ax,
    cmap="RdYlGn_r",
    xticklabels=models_plus,
    yticklabels=domain_labels,
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".0%",
    annot_kws={"fontsize": 10},
    square=False,
    linewidths=0,
    linecolor="none",
    cbar_kws={"label": "Fraction Significant"},
)
ax.set_title("Significance Rate by Domain and Model")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

fig.tight_layout()
save_figure(fig, "viz_14_significance_heatmap.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_14_significance_heatmap.png


In [18]:
# VIZ 15: Variance decomposition bar chart
fig, axes = plt.subplots(
    1, len(variance_metrics), figsize=(5 * len(variance_metrics), 5), sharey=True
)

factor_colors = {
    "model": "#1f77b4",
    "band": "#ff7f0e",
    "draw": "#2ca02c",
    "model_x_band": "#d62728",
    "Residual": "#7f7f7f",
}

for idx, metric in enumerate(variance_metrics):
    ax = axes[idx]
    sub = df_var[df_var["metric"] == metric]

    factors = sub["factor"].values
    eta2s = sub["eta_squared"].values
    colors = [factor_colors.get(f, "#999999") for f in factors]

    ax.bar(range(len(factors)), eta2s, color=colors, alpha=0.8)
    ax.set_xticks(range(len(factors)))
    ax.set_xticklabels(factors, rotation=45, ha="right", fontsize=8)
    ax.set_title(metric.replace("_", " ").title())
    if idx == 0:
        ax.set_ylabel("Eta-squared (proportion of variance)")

fig.suptitle("Variance Decomposition: Band vs Model vs Draw", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "viz_15_variance_decomposition.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_15_variance_decomposition.png


In [19]:
# VIZ 16: Structure-Function scatter plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

sf_pairs = [
    ("edge_fraction", "retention_ratio"),
    ("skip_fraction", "retention_ratio"),
    ("head_participation_rate", "retention_ratio"),
    ("edge_fraction", "circuit_kl_div"),
    ("skip_fraction", "circuit_kl_div"),
    ("attn_fraction", "circuit_accuracy"),
]

for idx, (s_met, f_met) in enumerate(sf_pairs):
    ax = axes[idx // 3][idx % 3]

    for model in MODELS:
        sub = df_merged[df_merged["model"] == model]
        ax.scatter(
            sub[s_met], sub[f_met], c=MODEL_COLORS[model], label=model, alpha=0.7, s=30
        )

    # Add correlation line
    valid = df_merged[[s_met, f_met]].dropna()
    if len(valid) > 5:
        rho, p = safe_spearmanr(valid[s_met].values, valid[f_met].values)
        ax.set_title(f"rho={rho:.2f}, p={p:.3f}", fontsize=10)

    ax.set_xlabel(s_met.replace("_", " "))
    ax.set_ylabel(f_met.replace("_", " "))
    if idx == 0:
        ax.legend(fontsize=7)

fig.suptitle("Structure-Function Correlations", fontsize=14, y=1.02)
fig.tight_layout()
save_figure(fig, "viz_16_structure_function.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_16_structure_function.png


In [20]:
# VIZ 17: Effect size forest plot for key tests
fig, ax = plt.subplots(figsize=(10, 12))

# Select tests with numeric effect sizes (exclude CV)
plotable = df_tests[
    (
        df_tests["effect_type"].isin(
            ["eta_squared", "rank_biserial", "cohens_d", "spearman_rho"]
        )
    )
    & (df_tests["effect_size"].notna())
].copy()

# Sort by absolute effect size
plotable["abs_effect"] = plotable["effect_size"].abs()
plotable = plotable.nlargest(30, "abs_effect")

y_pos = range(len(plotable))
colors = [
    BAND_COLORS.get(plotable.iloc[i]["model"], "#333333")
    if plotable.iloc[i]["model"] in BAND_COLORS
    else MODEL_COLORS.get(plotable.iloc[i]["model"], "#333333")
    for i in range(len(plotable))
]

# Color by significance
colors = [
    "#2ca02c" if plotable.iloc[i][sig_col] else "#d62728" for i in range(len(plotable))
]

ax.barh(y_pos, plotable["effect_size"].values, color=colors, alpha=0.7)
ax.set_yticks(y_pos)
labels = [
    f"{row['domain'].split('_', 1)[1]}: {row['hypothesis'][:40]}"
    for _, row in plotable.iterrows()
]
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel("Effect Size")
ax.set_title("Top 30 Effect Sizes (green=significant, red=not)")
ax.axvline(x=0, color="black", linestyle="-", linewidth=0.5)

fig.tight_layout()
save_figure(fig, "viz_17_effect_sizes.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_17_effect_sizes.png


In [21]:
# VIZ 18: Within vs between-band Jaccard comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(MODELS))
width = 0.35

within_means = []
within_stds = []
between_means = []
between_stds = []

for model in MODELS:
    wb = compute_within_between_jaccard(circuits, model)
    within_means.append(np.mean(wb["within"]) if wb["within"] else 0)
    within_stds.append(np.std(wb["within"]) if wb["within"] else 0)
    between_means.append(np.mean(wb["between"]) if wb["between"] else 0)
    between_stds.append(np.std(wb["between"]) if wb["between"] else 0)

ax.bar(
    x - width / 2,
    within_means,
    width,
    yerr=within_stds,
    label="Within-Band",
    color="#2ca02c",
    alpha=0.8,
    capsize=5,
)
ax.bar(
    x + width / 2,
    between_means,
    width,
    yerr=between_stds,
    label="Between-Band",
    color="#d62728",
    alpha=0.8,
    capsize=5,
)

ax.set_xticks(x)
ax.set_xticklabels(MODELS)
ax.set_ylabel("Jaccard Similarity")
ax.set_title("Within-Band vs Between-Band Circuit Similarity")
ax.legend()

fig.tight_layout()
save_figure(fig, "viz_18_within_between_jaccard.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_18_within_between_jaccard.png


In [22]:
# VIZ 19: Volcano plot (effect size vs -log10(p))
fig, ax = plt.subplots(figsize=(10, 7))

plot_df = df_tests[
    (df_tests["effect_size"].notna())
    & (df_tests["p_value"].notna())
    & (df_tests["p_value"] > 0)
].copy()

plot_df["neg_log_p"] = -np.log10(plot_df["p_value"])
plot_df["neg_log_p"] = plot_df["neg_log_p"].clip(upper=20)

# Color by domain
domain_colors = plt.cm.tab10(np.linspace(0, 1, len(plot_df["domain"].unique())))
domain_color_map = {
    d: domain_colors[i] for i, d in enumerate(plot_df["domain"].unique())
}

for domain in plot_df["domain"].unique():
    sub = plot_df[plot_df["domain"] == domain]
    ax.scatter(
        sub["effect_size"],
        sub["neg_log_p"],
        c=[domain_color_map[domain]],
        label=domain.replace("_", " "),
        alpha=0.7,
        s=30,
    )

# Significance threshold
ax.axhline(
    y=-np.log10(ALPHA), color="red", linestyle="--", alpha=0.5, label=f"p={ALPHA}"
)

ax.set_xlabel("Effect Size")
ax.set_ylabel("-log10(p-value)")
ax.set_title("Volcano Plot: Effect Size vs Significance")
ax.legend(fontsize=7, bbox_to_anchor=(1.05, 1), loc="upper left")

fig.tight_layout()
save_figure(fig, "viz_19_volcano_plot.png")

  Saved: LSC_circuit_analysis/02_Phase_Structural/outputs/viz/viz_19_volcano_plot.png


In [23]:
# Final output listing
print("\n" + "=" * 70)
print("ALL OUTPUTS")
print("=" * 70)
print("\nAnalysis CSVs:")
for f in sorted(ANALYSIS_DIR.glob("*.csv")):
    print(f"  {f.name}")
print("\nVisualizations:")
for f in sorted(VIZ_DIR.glob("*.png")):
    print(f"  {f.name}")


ALL OUTPUTS

Analysis CSVs:
  all_hypothesis_tests.csv
  band_jaccard.csv
  band_specific_edges.csv
  component_breakdown.csv
  deep_all_hypothesis_tests.csv
  deep_band_affinity.csv
  deep_band_signatures.csv
  deep_component_jaccard.csv
  deep_component_wiring.csv
  deep_degree_stats.csv
  deep_directed_containment.csv
  deep_draw_stability.csv
  deep_edge_sharing_raw.csv
  deep_graph_metrics.csv
  deep_head_clusters.csv
  deep_head_entropy.csv
  deep_head_universality.csv
  deep_hub_nodes.csv
  deep_input_edges.csv
  deep_layer_sensitivity.csv
  deep_layer_universal_fraction.csv
  deep_master_summary.csv
  deep_output_edges.csv
  deep_reliable_band_specific.csv
  deep_sharing_by_component.csv
  deep_sharing_profiles.csv
  deep_stability_vs_sharing.csv
  deep_universal_core_connectivity.csv
  draw_stability.csv
  edge_category_stats.csv
  edge_sharing.csv
  edge_stats.csv
  full_structure_data.csv
  head_participation.csv
  jaccard_summary.csv
  layer_flow_stats.csv
  master_structu